# 06 — Embeddings: ¿aporta una representación semántica del texto?

**Objetivo:** explorar si existe información textual en las fuentes institucionales que describa
aspectos profesionales/académicos del personal, y si una representación semántica (embeddings) de
ese texto puede **complementar** — no reemplazar — la representación estructurada usada en
`05_clustering` (`data/modeling/X_modelado.csv`).

**Restricciones del alcance (definidas en la consigna del proyecto):**

- No usar `IDPERSONA`, nombres, apellidos, URLs, referencias a archivos, ni metadata administrativa.
- Usar únicamente texto relevante para describir aspectos profesionales/académicos.
- Si el modelo de embeddings requiere API key / instalación adicional / acceso externo, **detenerse
  y preguntar** antes de instalar o usar credenciales — no se asume ni se inventa nada.


In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd().parents[1] if (Path.cwd().name == "06_embeddings") else Path.cwd()
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_MODELING = ROOT / "data" / "modeling"
DATA_EMBEDDINGS = ROOT / "data" / "embeddings"
DATA_EMBEDDINGS.mkdir(parents=True, exist_ok=True)

personas = pd.read_csv(DATA_MODELING / "personas_modelado.csv")
POBLACION = set(personas["IDPERSONA"])
print("Población de modelado (referencia para cobertura):", len(POBLACION))
print("Salidas de embeddings en:", DATA_EMBEDDINGS)


## 1. Columnas textuales disponibles en las fuentes institucionales

`data/features/*.csv` (usado en `03`-`05`) ya no tiene texto libre: todo fue agregado a conteos
numéricos por persona. El texto libre original vive en `data/processed/*.csv` (una fila por
publicación, capacitación, proyecto, etc., no por persona). Se revisan esas tablas en busca de
columnas de texto con contenido descriptivo real (no códigos, no IDs, no metadata administrativa).


In [ ]:
# Inventario de columnas candidatas identificadas por inspección de data/processed/*.csv
# (columnas de texto libre presentes en cada tabla, excluyendo IDs, fechas, referencias a archivos,
#  URLs, nombres/apellidos de personas y campos administrativos de auditoría/escalafón)
inventario_fuentes = pd.DataFrame([
    {"TABLA": "publicaciones.csv", "COLUMNA": "TITULO", "DESCRIPCION": "Título de publicación académica"},
    {"TABLA": "publicaciones.csv", "COLUMNA": "NOMBREREVISTA", "DESCRIPCION": "Nombre de la revista/venue (no describe el tema, sino el medio)"},
    {"TABLA": "proyecto_grado.csv", "COLUMNA": "NOMBRETRABAJOTITULACION", "DESCRIPCION": "Título del trabajo de titulación dirigido"},
    {"TABLA": "proyecto_grado.csv", "COLUMNA": "NOMBREPROGRAMA", "DESCRIPCION": "Programa académico del trabajo de titulación"},
    {"TABLA": "proyectos_investigacion_disponible.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Nombre del proyecto de investigación"},
    {"TABLA": "proyectos_investigacion_disponible.csv", "COLUMNA": "STRAREACAMPOAMPLIO / STRAREAFRASCATI / STRSUBAREAFRASCATI", "DESCRIPCION": "Área de conocimiento del proyecto (texto descriptivo)"},
    {"TABLA": "proyectos_investigacion_disponible.csv", "COLUMNA": "STRCAMPOESPECIFICO", "DESCRIPCION": "Contiene códigos numéricos ('1.0','2.0'...), no texto real (problema de calidad de datos)"},
    {"TABLA": "proyectos_vinculacion_disponible.csv", "COLUMNA": "NOMBREPROYECTO / NOMBREPROGRAMA", "DESCRIPCION": "Nombre del proyecto/programa de vinculación con la comunidad"},
    {"TABLA": "ponentes_todos.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Título de la ponencia/evento"},
    {"TABLA": "ponentes_todos.csv", "COLUMNA": "AREAACITACIONDESCRIPCION / TIPODESCRIPCION", "DESCRIPCION": "Códigos abreviados ('DI','PE','OT'), baja cobertura, no es texto descriptivo"},
    {"TABLA": "capacitaciones_todas.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Nombre de la capacitación tomada"},
    {"TABLA": "capacitaciones_todas.csv", "COLUMNA": "AREACAPACITACIONDESCRIPCION", "DESCRIPCION": "Códigos abreviados, baja cobertura"},
    {"TABLA": "certificados_todos.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Nombre de la certificación obtenida"},
    {"TABLA": "mencion_honor.csv", "COLUMNA": "NOMBREMENCION", "DESCRIPCION": "Categoría de la mención de honor (p.ej. 'Segunda mención'), no describe un tema"},
    {"TABLA": "experiencia_externa.csv", "COLUMNA": "CARGO", "DESCRIPCION": "Cargo ocupado en experiencia laboral externa"},
    {"TABLA": "experiencia_externa.csv", "COLUMNA": "INSTITUCION", "DESCRIPCION": "Nombre de la institución/empresa externa (identifica un organismo, no una competencia)"},
    {"TABLA": "carga_academica_disponible.csv", "COLUMNA": "NOMMATERIA", "DESCRIPCION": "Nombre de la materia impartida como docente"},
])
inventario_fuentes


## 2. Decisión: qué columnas se usan y por qué

Para cada tabla se mide, sobre la población de modelado (2213 personas), cuántas personas quedan
cubiertas por cada columna candidata, y se decide su inclusión con base en tres criterios: (a) si
describe realmente un **tema/competencia profesional o académica** (no un código, un medio o una
organización), (b) si tiene cobertura razonable, y (c) si no está prohibida por la consigna del
proyecto (IDs, nombres/apellidos, URLs, referencias a archivos, metadata administrativa).


In [ ]:
def cobertura(path, id_col, text_col, rename_id=None):
    df = pd.read_csv(DATA_PROCESSED / path, encoding="utf-8-sig")
    if rename_id:
        df = df.rename(columns={rename_id: id_col})
    df = df[df[id_col].isin(POBLACION) & df[text_col].notna()]
    return df[id_col].nunique(), len(df)

filas_decision = []

def registrar(fuente, tabla, columna, id_col, text_col, incluida, motivo, rename_id=None):
    n_personas, n_registros = cobertura(tabla, id_col, text_col, rename_id=rename_id)
    filas_decision.append({
        "FUENTE": fuente, "TABLA": tabla, "COLUMNA": text_col,
        "N_REGISTROS": n_registros, "N_PERSONAS_COBERTURA": n_personas,
        "PCT_POBLACION": round(100 * n_personas / len(POBLACION), 1),
        "INCLUIDA": incluida, "MOTIVO": motivo,
    })

registrar("PUBLICACION", "publicaciones.csv", "TITULO", "IDPERSONA", "TITULO",
          True, "Título describe directamente el tema de investigación")
registrar("PROYECTO_GRADO_DIRIGIDO", "proyecto_grado.csv", "NOMBRETRABAJOTITULACION", "IDPERSONA", "NOMBRETRABAJOTITULACION",
          True, "Título de tesis dirigida describe área de especialidad", rename_id="IDDIRECTOR")
registrar("PROYECTO_INVESTIGACION", "proyectos_investigacion_disponible.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Nombre de proyecto describe el tema investigado")
registrar("PROYECTO_VINCULACION", "proyectos_vinculacion_disponible.csv", "NOMBREPROYECTO", "IDPERSONA", "NOMBREPROYECTO",
          True, "Nombre de proyecto de vinculación describe el tema/comunidad de trabajo")
registrar("PONENCIA", "ponentes_todos.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Título de ponencia describe el tema presentado")
registrar("CAPACITACION", "capacitaciones_todas.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Nombre de capacitación describe el área de formación continua; cobertura muy alta")
registrar("CERTIFICACION", "certificados_todos.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Nombre de certificación describe una competencia adquirida")
registrar("EXPERIENCIA_EXTERNA_CARGO", "experiencia_externa.csv", "CARGO", "IDPERSONA", "CARGO",
          True, "Cargo externo describe rol/dominio profesional fuera de ESPOL")
registrar("MATERIA_IMPARTIDA", "carga_academica_disponible.csv", "NOMMATERIA", "IDPERSONA", "NOMMATERIA",
          True, "Nombre de materia impartida describe el área de docencia")
registrar("MENCION_HONOR", "mencion_honor.csv", "NOMBREMENCION", "IDPERSONA", "NOMBREMENCION",
          False, "Es una categoría de reconocimiento ('Diploma de honor'), no un tema/competencia")

decision_fuentes = pd.DataFrame(filas_decision).sort_values("PCT_POBLACION", ascending=False)
decision_fuentes


**Columnas descartadas explícitamente** (no llegan a la tabla de decisión porque no califican
como texto descriptivo, o violan las restricciones del alcance):

- `NOMBREREVISTA` (publicaciones) y `INSTITUCION` (experiencia externa): nombran un medio/organismo,
  no una competencia o tema — quedarían más cerca de "metadata administrativa" que de contenido
  profesional/académico.
- `STRCAMPOESPECIFICO` (proyectos de investigación): pese al nombre de la columna, contiene códigos
  numéricos como texto ('1.0', '2.0'), no descripciones — problema de calidad de datos, no una
  fuente de texto real.
- `AREAACITACIONDESCRIPCION`, `TIPODESCRIPCION` (ponentes) y `AREACAPACITACIONDESCRIPCION`
  (capacitaciones): son códigos abreviados ('DI', 'PE', 'OT') con cobertura muy baja, no texto
  descriptivo utilizable.
- Todo identificador (`IDPERSONA`, `IDCAPACITACION`, etc.), nombre/apellido de persona
  (`NOMBRES`, `APELLIDOS` en `heteroevaluacion_disponible.csv`), URL (`URLPUBLICACION`), referencia
  a archivo (`REFARCHIVO*`, `NAMEARCHDOC`) y campo administrativo de auditoría/escalafón
  (`ENESCALAFON`, `REVISADOPARAESCALAFON`, `IDUSUARIO`, `INGRESORRHH`, `ORIGENINGRESO`,
  `FECHASUBIDAARCHIVO`, etc.) — excluidos por instrucción explícita del proyecto.
- No se encontraron campos de comentarios/evaluaciones cualitativas de CENACAD en las tablas
  disponibles en `data/processed/` (solo el promedio numérico de heteroevaluación, ya incorporado
  en `X_modelado` como `PROMEDIO_HETEROEVALUACION`); si existen en otra fuente institucional no
  integrada aún, deben añadirse como una fuente adicional en una futura iteración.


## 3. Construcción del corpus textual por persona

Se construye una tabla de detalle (una fila por **registro** de texto: una publicación, una
capacitación, etc., asociada a su `IDPERSONA` y a la fuente de la que proviene) y, a partir de ella,
un resumen de cobertura por persona. Este detalle es el insumo para la sección 4 de este notebook y
para el cálculo de embeddings en una iteración siguiente — **no** es el archivo final de embeddings
(ese no debe contener texto, ver sección 5).

Cada texto conserva su origen (`FUENTE`) para poder ponderar o filtrar por tipo de fuente más
adelante si fuera necesario.


In [ ]:
def cargar(path):
    return pd.read_csv(DATA_PROCESSED / path, encoding="utf-8-sig")

registros = []

def agregar(df, id_col, fuente, texto_fn, rename_id=None):
    if rename_id:
        df = df.rename(columns={rename_id: id_col})
    df = df[df[id_col].isin(POBLACION)]
    for _, r in df.iterrows():
        texto = texto_fn(r)
        if texto and pd.notna(texto) and str(texto).strip():
            registros.append((int(r[id_col]), fuente, str(texto).strip()))

agregar(cargar("publicaciones.csv"), "IDPERSONA", "PUBLICACION",
        lambda r: r["TITULO"] if pd.notna(r["TITULO"]) else None)

agregar(cargar("proyecto_grado.csv"), "IDPERSONA", "PROYECTO_GRADO_DIRIGIDO",
        lambda r: (r["NOMBRETRABAJOTITULACION"] + (f" ({r['NOMBREPROGRAMA']})" if pd.notna(r.get("NOMBREPROGRAMA")) else ""))
                  if pd.notna(r["NOMBRETRABAJOTITULACION"]) else None,
        rename_id="IDDIRECTOR")

agregar(cargar("proyectos_investigacion_disponible.csv"), "IDPERSONA", "PROYECTO_INVESTIGACION",
        lambda r: " — ".join([str(r["NOMBRE"])] + [str(r[c]) for c in
                  ["STRAREACAMPOAMPLIO", "STRAREAFRASCATI", "STRSUBAREAFRASCATI"] if pd.notna(r.get(c))])
                  if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("proyectos_vinculacion_disponible.csv"), "IDPERSONA", "PROYECTO_VINCULACION",
        lambda r: r["NOMBREPROYECTO"] + (f" — {r['NOMBREPROGRAMA']}" if pd.notna(r.get("NOMBREPROGRAMA")) else "")
                  if pd.notna(r["NOMBREPROYECTO"]) else None)

agregar(cargar("ponentes_todos.csv"), "IDPERSONA", "PONENCIA",
        lambda r: r["NOMBRE"] if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("capacitaciones_todas.csv"), "IDPERSONA", "CAPACITACION",
        lambda r: r["NOMBRE"] if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("certificados_todos.csv"), "IDPERSONA", "CERTIFICACION",
        lambda r: r["NOMBRE"] if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("experiencia_externa.csv"), "IDPERSONA", "EXPERIENCIA_EXTERNA_CARGO",
        lambda r: r["CARGO"] if pd.notna(r["CARGO"]) else None)

df_materias = cargar("carga_academica_disponible.csv")
df_materias = df_materias[df_materias["IDPERSONA"].isin(POBLACION) & df_materias["NOMMATERIA"].notna()]
df_materias = df_materias.drop_duplicates(subset=["IDPERSONA", "NOMMATERIA"])
agregar(df_materias, "IDPERSONA", "MATERIA_IMPARTIDA", lambda r: r["NOMMATERIA"])

corpus_detalle = pd.DataFrame(registros, columns=["IDPERSONA", "FUENTE", "TEXTO"])
corpus_detalle = corpus_detalle.drop_duplicates(subset=["IDPERSONA", "FUENTE", "TEXTO"]).reset_index(drop=True)

print("Registros de texto totales:", len(corpus_detalle))
print("Personas con al menos un registro de texto:", corpus_detalle["IDPERSONA"].nunique(),
      f"de {len(POBLACION)} ({100*corpus_detalle['IDPERSONA'].nunique()/len(POBLACION):.1f}%)")
corpus_detalle["FUENTE"].value_counts()


In [ ]:
corpus_detalle.to_csv(DATA_EMBEDDINGS / "corpus_texto_detalle.csv", index=False)
print("Guardado:", DATA_EMBEDDINGS / "corpus_texto_detalle.csv", "-", corpus_detalle.shape)


## 4. Diagnóstico de cobertura del corpus

Se resume, por persona, cuántos registros de texto tiene y de cuántas fuentes distintas, **sin**
guardar el texto en este resumen (el texto vive únicamente en `corpus_texto_detalle.csv`, que es
insumo de trabajo, no un producto final).


In [ ]:
resumen_personas = (
    corpus_detalle.groupby("IDPERSONA")
    .agg(N_REGISTROS_TEXTO=("TEXTO", "size"), N_FUENTES_DISTINTAS=("FUENTE", "nunique"))
    .reset_index()
)
resumen_personas["LONGITUD_TOTAL_CARACTERES"] = (
    corpus_detalle.groupby("IDPERSONA")["TEXTO"].apply(lambda s: s.str.len().sum()).values
)

cobertura_completa = personas[["IDPERSONA"]].merge(resumen_personas, on="IDPERSONA", how="left")
cobertura_completa[["N_REGISTROS_TEXTO", "N_FUENTES_DISTINTAS", "LONGITUD_TOTAL_CARACTERES"]] = (
    cobertura_completa[["N_REGISTROS_TEXTO", "N_FUENTES_DISTINTAS", "LONGITUD_TOTAL_CARACTERES"]].fillna(0)
)
cobertura_completa.to_csv(DATA_EMBEDDINGS / "cobertura_texto_personas.csv", index=False)

sin_texto = (cobertura_completa["N_REGISTROS_TEXTO"] == 0).sum()
print(f"Personas sin ningún registro de texto: {sin_texto} ({100*sin_texto/len(cobertura_completa):.1f}%)")
cobertura_completa[["N_REGISTROS_TEXTO", "N_FUENTES_DISTINTAS", "LONGITUD_TOTAL_CARACTERES"]].describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(cobertura_completa["N_REGISTROS_TEXTO"].clip(upper=100), bins=40, color="#4C72B0")
axes[0].set_title("Registros de texto por persona (recortado en 100)")
axes[0].set_xlabel("N° de registros de texto")

fuente_counts = corpus_detalle["FUENTE"].value_counts()
axes[1].barh(fuente_counts.index[::-1], fuente_counts.values[::-1], color="#55A868")
axes[1].set_title("Registros de texto por fuente")
axes[1].set_xlabel("N° de registros")

plt.tight_layout()
plt.show()


**Lectura del diagnóstico:** el corpus cubre ~99% de la población de modelado con al menos un
registro de texto, dominado en volumen por `CAPACITACION` (más de la mitad de los registros) seguido
de `PUBLICACION`, `PROYECTO_INVESTIGACION`, `EXPERIENCIA_EXTERNA_CARGO` y `PROYECTO_GRADO_DIRIGIDO`.
Esto confirma que **sí existe suficiente texto institucional relevante** como para justificar
explorar una representación semántica — pero también advierte que, al calcular los embeddings, no
conviene concatenar todo el texto de una persona en un solo string (algunas personas superan 10,000
palabras): la mayoría de los modelos de embeddings truncan a unos pocos cientos de tokens. El diseño
más apropiado es **calcular un embedding por registro de texto y luego agregarlo (p.ej. promedio)
por persona**, posiblemente ponderando por fuente para que `CAPACITACION` no domine desproporcionadamente
solo por su volumen. Esta decisión de diseño se aplicará en la siguiente iteración, junto con el
modelo de embeddings elegido.


## 5. Modelo de embeddings — necesito una decisión antes de continuar

Con el corpus ya construido (`data/embeddings/corpus_texto_detalle.csv`), el siguiente paso técnico
es calcular un vector de embedding por cada texto y agregarlo por persona. Antes de hacerlo, y
siguiendo la instrucción explícita del proyecto, **me detengo aquí** porque cualquier modelo de
embeddings razonable para este corpus (mayormente en español, con términos técnicos/académicos)
requiere una decisión sobre instalación y/o acceso externo:

### Opción A — Modelo local/open-source (sin API key)

- **Servicio/modelo:** un modelo de `sentence-transformers` multilingüe, p. ej.
  `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (rápido, ~118M parámetros) o
  `intfloat/multilingual-e5-small` (buen desempeño multilingüe, tamaño similar).
- **API key:** ninguna.
- **Variable de entorno:** ninguna.
- **Dependencia a instalar:** `pip install sentence-transformers` (instala también `torch` si no
  está presente). La primera ejecución descarga los pesos del modelo (~100-500 MB) desde Hugging
  Face Hub, lo que requiere acceso a internet **una sola vez**; el cómputo en sí corre localmente y
  el texto no sale de la máquina.
- **Ventaja para este proyecto:** coherente con el principio de privacidad/gobernanza de datos
  institucionales del contexto permanente del proyecto (los datos de personal de ESPOL no se envían
  a un tercero).

### Opción B — API de embeddings de un proveedor externo

- **Servicio/modelo:** p. ej. OpenAI (`text-embedding-3-small`), Voyage AI (`voyage-3` /
  `voyage-multilingual-2`) o Cohere (`embed-multilingual-v3.0`).
- **API key:** sí, del proveedor elegido.
- **Variable de entorno:** p. ej. `OPENAI_API_KEY`, `VOYAGE_API_KEY` o `COHERE_API_KEY` (según el
  proveedor).
- **Dónde obtenerla:** en el panel de desarrollador del proveedor (p. ej.
  platform.openai.com/api-keys, dashboard.voyageai.com, dashboard.cohere.com).
- **Dependencia a instalar:** el SDK del proveedor (`openai`, `voyageai` o `cohere`).
- **Consideración de privacidad:** el texto (títulos de tesis, proyectos, cargos, etc., asociado
  indirectamente a personal de ESPOL) saldría hacia un servicio externo — el contexto permanente del
  proyecto pide considerar explícitamente privacidad, gobernanza y normativa ecuatoriana antes de
  hacer esto.

**No voy a instalar nada ni a inventar una API key.** Si eliges la Opción B, dime qué proveedor
prefieres, crea la variable de entorno correspondiente con tu API key, y avísame cuando esté lista.
Si eliges la Opción A, solo necesito tu confirmación para instalar `sentence-transformers`.


## 6. Cálculo de embeddings (Opción A confirmada: modelo local `sentence-transformers`)

Se usa `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (multilingüe, 384 dimensiones,
sin API key, corre localmente). Para eficiencia se codifican únicamente los **textos únicos** del
corpus (muchos nombres de capacitación/certificación se repiten entre personas) y luego se
distribuye cada embedding a todos los registros que comparten ese texto.


In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

textos_unicos = corpus_detalle["TEXTO"].astype(str).unique().tolist()
print(f"Codificando {len(textos_unicos)} textos únicos...")

embeddings_unicos = model.encode(
    textos_unicos, batch_size=128, show_progress_bar=True, normalize_embeddings=True,
)
DIM_EMBEDDING = embeddings_unicos.shape[1]
print("Embeddings calculados:", embeddings_unicos.shape)

texto_to_idx = {t: i for i, t in enumerate(textos_unicos)}
corpus_detalle["EMB_IDX"] = corpus_detalle["TEXTO"].map(texto_to_idx)


## 7. Agregación por persona

Se agregan los embeddings a nivel de persona en dos variantes, para poder comparar el efecto de la
elección de agregación (ver advertencia de la sección 4 sobre el volumen desigual entre fuentes):

- **`naive`**: promedio simple de todos los registros de texto de la persona (las fuentes con más
  registros, como `CAPACITACION`, pesan más).
- **`balanceada`** (la que se usa como salida final): promedio por fuente y luego promedio de esos
  promedios entre fuentes — cada fuente aporta lo mismo sin importar cuántos registros tenga esa
  persona en ella.

Ambos vectores se renormalizan a norma unitaria al final (la media de vectores unitarios no es, en
general, unitaria).


In [ ]:
def normalizar(v):
    norma = np.linalg.norm(v)
    return v / norma if norma > 0 else v

emb_matrix = embeddings_unicos  # (n_textos_unicos, DIM)

# Variante naive: promedio simple de todos los registros por persona
naive = (
    corpus_detalle.groupby("IDPERSONA")["EMB_IDX"]
    .apply(lambda idxs: normalizar(emb_matrix[idxs.values].mean(axis=0)))
)

# Variante balanceada: promedio por fuente, luego promedio entre fuentes
por_fuente = (
    corpus_detalle.groupby(["IDPERSONA", "FUENTE"])["EMB_IDX"]
    .apply(lambda idxs: emb_matrix[idxs.values].mean(axis=0))
)
balanceada = (
    por_fuente.groupby("IDPERSONA")
    .apply(lambda s: normalizar(np.mean(np.stack(s.values), axis=0)))
)

ids_con_texto = naive.index.to_numpy()
naive_matrix = np.stack(naive.values)
balanceada_matrix = np.stack(balanceada.values)

sim_naive_balanceada = (naive_matrix * balanceada_matrix).sum(axis=1)  # coseno (vectores unitarios)
print(f"Personas con embedding: {len(ids_con_texto)} / {len(POBLACION)}")
print("Similaridad coseno entre variante naive y balanceada por persona:")
print(pd.Series(sim_naive_balanceada).describe())


La similaridad entre ambas variantes es alta pero no perfecta — confirma que la elección de
agregación sí mueve el vector resultante, como se anticipó. Se usa la **variante balanceada** como
salida oficial, para que el volumen de `CAPACITACION` no domine desproporcionadamente el perfil
semántico de cada persona.


In [ ]:
embeddings_personas = pd.DataFrame(
    balanceada_matrix, columns=[f"E_{i:03d}" for i in range(DIM_EMBEDDING)],
)
embeddings_personas.insert(0, "IDPERSONA", ids_con_texto)

embeddings_personas.to_csv(DATA_EMBEDDINGS / "embeddings_personas.csv", index=False)

sin_embedding = sorted(POBLACION - set(ids_con_texto))
print(f"Guardado: {DATA_EMBEDDINGS / 'embeddings_personas.csv'} — {embeddings_personas.shape}")
print(f"Personas SIN embedding (sin ningún registro de texto): {len(sin_embedding)}")
print("IDs:", sin_embedding[:10], "..." if len(sin_embedding) > 10 else "")


In [ ]:
metadata_embeddings = pd.DataFrame([{
    "MODELO": MODEL_NAME,
    "DIMENSIONES": DIM_EMBEDDING,
    "METODO_POOLING": "promedio por fuente, luego promedio entre fuentes (balanceado), renormalizado",
    "N_TEXTOS_UNICOS_CODIFICADOS": len(textos_unicos),
    "N_REGISTROS_TEXTO_TOTAL": len(corpus_detalle),
    "N_PERSONAS_CON_EMBEDDING": len(ids_con_texto),
    "N_PERSONAS_SIN_EMBEDDING": len(sin_embedding),
    "FUENTES_INCLUIDAS": ", ".join(sorted(corpus_detalle["FUENTE"].unique())),
}])
metadata_embeddings.to_csv(DATA_EMBEDDINGS / "embeddings_metadata.csv", index=False)
metadata_embeddings.T


### Validación cualitativa: vecinos más cercanos en el espacio semántico

Antes de cualquier análisis cuantitativo, una revisión cualitativa rápida: para una persona con
suficiente texto (varias publicaciones/proyectos), ¿sus vecinos más cercanos por similitud coseno
tienen un perfil temático parecido? Esto no es una prueba formal, es una verificación de sentido
común de que el embedding capturó algo razonable.


In [ ]:
cov = cobertura_completa.set_index("IDPERSONA")
candidatos = cov[cov["N_REGISTROS_TEXTO"] >= 15].index
persona_ejemplo = int(pd.Series(sorted(set(candidatos) & set(ids_con_texto))).sample(1, random_state=7).iloc[0])

idx_ejemplo = list(ids_con_texto).index(persona_ejemplo)
sims = balanceada_matrix @ balanceada_matrix[idx_ejemplo]
orden = np.argsort(-sims)
vecinos = [ids_con_texto[i] for i in orden[1:6]]

def resumen_texto(idp, max_chars=200):
    txts = corpus_detalle.loc[corpus_detalle["IDPERSONA"] == idp, "TEXTO"].tolist()
    return (" | ".join(txts))[:max_chars]

print(f"Persona de referencia {persona_ejemplo}:")
print(" ", resumen_texto(persona_ejemplo))
print("\nVecinos más cercanos (similitud coseno):")
for v in vecinos:
    print(f"  [{sims[list(ids_con_texto).index(v)]:.3f}] {v}: {resumen_texto(v)}")


## 8. ¿Los embeddings identifican una estructura distinta a la de `05_clustering`?

Se agrupa el espacio de embeddings por sí solo (K-Means, igual que en `05`) y se compara contra los
clusters estructurales ya obtenidos (`data/clustering/clusters_personas.csv`). La comparación se
limita a las 2196 personas que tienen al menos un embedding (las 17 sin texto quedan fuera de esta
comparación puntual, no del resto del proyecto).


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA

clusters_estructurales = pd.read_csv(ROOT / "data" / "clustering" / "clusters_personas.csv")

X_emb = balanceada_matrix  # ya normalizado por fila
RANDOM_STATE = 42

filas_emb_k = []
for k in range(3, 9):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_emb)
    filas_emb_k.append({"K": k, "SILHOUETTE": silhouette_score(X_emb, labels)})
pd.DataFrame(filas_emb_k)


Se usa **K=5** para los embeddings (el mismo K que la solución estructural final), de modo que
la comparación entre ambas particiones sea directa y no esté confundida por un número distinto de
grupos.


In [ ]:
labels_emb = KMeans(n_clusters=5, n_init=10, random_state=RANDOM_STATE).fit_predict(X_emb)

emb_clusters_df = pd.DataFrame({"IDPERSONA": ids_con_texto, "CLUSTER_EMBEDDING": labels_emb})
comparacion = emb_clusters_df.merge(clusters_estructurales, on="IDPERSONA", how="inner")
comparacion = comparacion.rename(columns={"CLUSTER": "CLUSTER_ESTRUCTURAL"})

ari = adjusted_rand_score(comparacion["CLUSTER_ESTRUCTURAL"], comparacion["CLUSTER_EMBEDDING"])
print(f"Personas comparadas: {len(comparacion)}")
print(f"ARI (estructural vs. embeddings, ambos K=5): {ari:.3f}")
print()
tabla_cruzada = pd.crosstab(comparacion["CLUSTER_ESTRUCTURAL"], comparacion["CLUSTER_EMBEDDING"])
tabla_cruzada


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

pca_emb = PCA(n_components=2, random_state=RANDOM_STATE)
X_emb_pca = pca_emb.fit_transform(X_emb)

paleta5 = sns.color_palette("Set2", 5)
merged_pca = pd.DataFrame(X_emb_pca, columns=["PC1", "PC2"])
merged_pca["IDPERSONA"] = ids_con_texto
merged_pca = merged_pca.merge(comparacion, on="IDPERSONA", how="left")

for c in range(5):
    m = merged_pca["CLUSTER_EMBEDDING"] == c
    axes[0].scatter(merged_pca.loc[m, "PC1"], merged_pca.loc[m, "PC2"], s=10, alpha=0.6, color=paleta5[c], label=f"C{c}")
axes[0].set_title("Espacio de embeddings (PCA 2D)\ncoloreado por cluster DE EMBEDDINGS")
axes[0].legend(fontsize=8, markerscale=2)

for c in range(5):
    m = merged_pca["CLUSTER_ESTRUCTURAL"] == c
    axes[1].scatter(merged_pca.loc[m, "PC1"], merged_pca.loc[m, "PC2"], s=10, alpha=0.6, color=paleta5[c], label=f"C{c}")
axes[1].set_title("Espacio de embeddings (PCA 2D)\ncoloreado por cluster ESTRUCTURAL (05)")
axes[1].legend(fontsize=8, markerscale=2)

plt.tight_layout()
plt.show()


**Lectura:** un ARI cercano a 0 indicaría que los embeddings capturan una estructura
prácticamente independiente de la estructural (alta complementariedad); un ARI cercano a 1
indicaría que agrupan a las personas de forma casi idéntica (redundancia, poco aporte adicional).
El valor obtenido (ver celda anterior) y el gráfico de la derecha — que muestra si los clusters
estructurales forman regiones reconocibles o aparecen mezclados en el espacio semántico — se toman
en conjunto para la conclusión de la sección 9.


## 9. ¿Aportan los embeddings algo que las features estructuradas no capturan?

Además del ARI global, conviene revisar si el espacio semántico distingue **dentro** de un mismo
cluster estructural — por ejemplo, dos docentes de "alta carga docente" (mismo Cluster 3 en `05`)
pueden enseñar/investigar en dominios completamente distintos (ingeniería vs. ciencias sociales), y
esa diferencia de dominio es justamente lo que las 100 columnas de `X_modelado` no capturan (son
conteos de actividad, no de contenido/tema).


In [ ]:
# Dispersión semántica dentro de cada cluster estructural: similitud coseno promedio
# entre pares de personas del mismo cluster vs. pares de clusters distintos.
from itertools import combinations
rng = np.random.default_rng(42)

def similitud_promedio_pares(indices, n_pares=300):
    if len(indices) < 2:
        return np.nan
    pares = rng.choice(len(indices), size=(min(n_pares, len(indices) * (len(indices) - 1) // 2), 2))
    pares = pares[pares[:, 0] != pares[:, 1]]
    sims = [float(X_emb[indices[i]] @ X_emb[indices[j]]) for i, j in pares]
    return np.mean(sims)

idx_por_cluster_estructural = {
    c: [list(ids_con_texto).index(i) for i in comparacion.loc[comparacion["CLUSTER_ESTRUCTURAL"] == c, "IDPERSONA"]]
    for c in sorted(comparacion["CLUSTER_ESTRUCTURAL"].unique())
}

filas_dispersión = []
for c, idxs in idx_por_cluster_estructural.items():
    filas_dispersión.append({
        "CLUSTER_ESTRUCTURAL": c,
        "N_PERSONAS": len(idxs),
        "SIMILITUD_SEMANTICA_INTRA_CLUSTER": similitud_promedio_pares(idxs),
    })

todos_los_idx = list(range(len(ids_con_texto)))
sim_global = similitud_promedio_pares(todos_los_idx, n_pares=500)

dispersión_df = pd.DataFrame(filas_dispersión)
print(f"Similitud semántica promedio entre dos personas cualesquiera (referencia global): {sim_global:.3f}")
dispersión_df


**Interpretación:** si la similitud semántica promedio **dentro** de cada cluster estructural
fuera muy superior a la similitud global de referencia, significaría que el clustering estructural
(basado en volumen/tipo de actividad) ya agrupa implícitamente a personas con temas afines — los
embeddings aportarían poco. Si en cambio la similitud intra-cluster es parecida a la global, cada
cluster estructural mezcla temas muy distintos entre sí, y una representación semántica sí aportaría
una dimensión de información **adicional** (a qué se dedica temáticamente cada persona) que
`X_modelado` no captura.

**Resultado obtenido:** ARI = **0.116** entre la partición estructural y la de embeddings (ambas
K=5) — muy lejos de 1, señal de particiones prácticamente independientes. La similitud semántica
intra-cluster (referencia global: 0.693) es:

- Cluster 1 (perfil administrativo): 0.786 — el más homogéneo temáticamente, esperable dado que el
  vocabulario administrativo/laboral es más acotado que el académico.
- Clusters 0, 2 y 3 (perfiles docentes): 0.72-0.75 — apenas por encima del global, es decir, dentro
  de cada uno de estos clusters conviven temas/disciplinas bastante distintos.
- Cluster 4 (ingreso reciente): 0.578 — el más heterogéneo temáticamente, incluso por debajo del
  promedio global: el criterio que define este cluster (poca antigüedad) no tiene ninguna relación
  con el área de conocimiento de la persona.

**Conclusión de esta exploración (06):**

- Los embeddings **sí capturan información distinta** a la estructural: el ARI de 0.116 confirma
  particiones mayormente independientes, y salvo el caso parcial del cluster administrativo, la
  similitud intra-cluster estructural no supera de forma relevante a la similitud global — es decir,
  los clusters estructurales (definidos por volumen/tipo de actividad: docencia, investigación,
  administración, antigüedad) en general NO predicen bien de qué tema/disciplina trata el trabajo de
  cada persona. Ambas representaciones son **complementarias**, no redundantes: la estructural
  describe *cuánto y qué tipo* de actividad tiene una persona; la semántica describe *sobre qué*
  trata esa actividad.
- **No se concatena una matriz combinada (100 + 384 dimensiones) en este notebook.** Aunque hay
  evidencia de complementariedad, concatenar directamente estructurado + embeddings sin un análisis
  adicional (p. ej. reducir la dimensionalidad de los embeddings para no dominar la distancia
  euclídea frente a las 100 columnas, decidir una ponderación relativa, y re-evaluar
  estabilidad/interpretabilidad del clustering resultante) no está metodológicamente justificado
  todavía — así lo pide la consigna del proyecto. Queda documentado como una línea de trabajo futura
  concreta, no como algo ya resuelto.
- **Uso recomendado en el corto plazo:** mantener ambas representaciones **separadas**. Los perfiles
  estructurales de `05_clustering` siguen siendo la base para el dashboard (`07`); los embeddings de
  este notebook pueden usarse de forma independiente para funciones complementarias como "buscar
  personas con experiencia temática similar a X" (búsqueda semántica) dentro de cada perfil
  estructural, sin fusionar ambas matrices.

## 10. Resumen

**Archivos generados en `data/embeddings/`:**

| Archivo | Contenido |
|---|---|
| `corpus_texto_detalle.csv` | `IDPERSONA, FUENTE, TEXTO` — insumo de trabajo (una fila por registro de texto) |
| `cobertura_texto_personas.csv` | Cobertura de texto por persona, sin texto (solo conteos) |
| `embeddings_personas.csv` | `IDPERSONA` + 384 columnas `E_000..E_383` (embedding balanceado por fuente, normalizado) |
| `embeddings_metadata.csv` | Modelo usado, dimensiones, método de agregación, cobertura |

**Decisiones tomadas:**

1. Se identificaron 9 fuentes de texto profesional/académico relevantes (publicaciones, tesis
   dirigidas, proyectos de investigación/vinculación, ponencias, capacitaciones, certificaciones,
   cargos externos, materias impartidas) y se excluyeron explícitamente columnas que eran códigos,
   nombres de medios/organizaciones, categorías de reconocimiento, o metadata administrativa.
2. Se usó el modelo local `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (Opción A,
   confirmada por el usuario) — sin API key, sin enviar datos a terceros.
3. Se agregó a nivel de persona con **promedio balanceado por fuente** (no promedio simple), para
   que el volumen de `CAPACITACION` no domine el vector semántico.
4. Se comparó la estructura de los embeddings contra los clusters de `05_clustering` y se concluyó
   que son **complementarios**: no se fusionan automáticamente en una sola matriz, siguiendo la
   instrucción explícita del proyecto de no concatenar sin justificación metodológica.

**Limitaciones:**

- 17 personas (0.8%) no tienen ningún registro de texto y por lo tanto no tienen embedding — deben
  tratarse explícitamente (excluir o imputar) en cualquier uso posterior de `embeddings_personas.csv`.
- El corpus está dominado en volumen por `CAPACITACION`; se corrigió con agregación balanceada, pero
  sigue siendo la fuente con más cobertura poblacional y más peso en la elección de qué "temas"
  aparecen representados.
- No se evaluaron alternativas de modelo (p. ej. `multilingual-e5-small`) ni de pooling más
  sofisticadas (p. ej. ponderar por recencia o por tipo de fuente con pesos distintos a 1); son
  posibles mejoras futuras si se decide seguir esta línea.
